In [69]:
import pyodbc
from pyodbc import Error
from typing import List, Tuple
from datetime import datetime
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [2]:

# Global Configuration
SERVER = "localhost\\SQLEXPRESS"       # Change to your server name/instance
DRIVER = "SQL Server" # Ensure this driver is installed on your OS


def connection(db_name: str = "master") -> pyodbc.Connection:
    """Creates a connection using Windows Integrated Security (Trusted_Connection)."""
    conn_str = (
        f"DRIVER={{{DRIVER}}};"
        f"SERVER={SERVER};"
        f"DATABASE={db_name};"
        f"Trusted_Connection=yes;"  # <--- Crucial flag for Windows Auth
    )
    # autocommit=True is mandatory for running administrative tasks like CREATE DATABASE
    return pyodbc.connect(conn_str, autocommit=True)


def create_database(db_name: str) -> None:
    """Connects to master database and creates the target database if missing."""
    check_query = "SELECT database_id FROM sys.databases WHERE name = ?"
    create_query = f"CREATE DATABASE [{db_name}]"
    
    try:
        with connection(db_name="master") as conn:
            with conn.cursor() as cursor:
                cursor.execute(check_query, (db_name,))
                if cursor.fetchone():
                    print(f"Database '{db_name}' already exists.")
                    return
                else:
                    print(f"Database missing. Generating '{db_name}'...")
                cursor.execute(create_query)
                print(f"Database '{db_name}' successfully built!")
                
    except Error as e:
        print(f"Database creation failed: {e}")


def create_tables(db_name: str) -> None:
    # """Connects directly to the new database to execute table definitions."""
    # table_query = """
    # IF NOT EXISTS (SELECT * FROM sys.objects WHERE object_id = OBJECT_ID(N'[dbo].[Employees]') AND type in (N'U'))
    # BEGIN
    #     CREATE TABLE Employees (
    #         EmpID INT IDENTITY(1,1) PRIMARY KEY,
    #         FullName NVARCHAR(100) NOT NULL,
    #         Department NVARCHAR(50) NULL,
    #         HireDate DATE DEFAULT GETDATE()
    #     )
    # END
    # """
    table_query = """
        IF NOT EXISTS (SELECT * FROM sys.objects WHERE object_id = OBJECT_ID(N'[dbo].[Users]') AND type in (N'U'))
        BEGIN
            CREATE TABLE Users (
                UserID INT IDENTITY(1,1) PRIMARY KEY,
                Username VARCHAR(50) NOT NULL,
                Email VARCHAR(100) NOT NULL,
                CreatedAt DATETIME DEFAULT GETDATE()
            )
            END;
        """
    
    try:
        with connection(db_name=db_name) as conn:
            with conn.cursor() as cursor:
                cursor.execute(table_query)
                print("Table structures generated/verified.")
                
    except Error as e:
        print(f"Schema generation failed: {e}")


if __name__ == "__main__":
    TARGET_DATABASE = "CommercialDB"
    create_database(TARGET_DATABASE)
    
    create_tables(TARGET_DATABASE)


Database 'CommercialDB' already exists.
Table structures generated/verified.


In [ ]:

SERVER = "localhost\\SQLEXPRESS"
DRIVER = "SQL Server"
DATABASE = "SalesDB"

def get_db_connection() -> pyodbc.Connection:
    conn_str = (
        f"DRIVER={{{DRIVER}}};"
        f"SERVER={SERVER};"
        f"DATABASE={DATABASE};"
        f"Trusted_Connection=yes;"
    )
    return pyodbc.connect(conn_str)


def insert_customers(sql_query: str, customer_list: List[Tuple[int, str, str, int]]) -> None:
    # sql_query = "INSERT INTO Customers (CustomerID, CustomerName, Email, PhoneNumber) VALUES (?, ?, ?, ?)"
    
    try:
        with get_db_connection() as conn:
            with conn.cursor() as cursor:
                cursor.fast_executemany = True
                cursor.executemany(sql_query, customer_list)
                conn.commit()
                print(f"Successfully bulk-inserted {len(customer_list)} records.")
                
    except Error as e:
        print(f"Error during bulk insertion: {e}")

def insert_products(product_list: List[Tuple[int, str, str, float]]) -> None:
    sql_query = "INSERT INTO Products (ProductID, ProductName, Description, Price) VALUES (?, ?, ?, ?)"
    
    try:
        with get_db_connection() as conn:
            with conn.cursor() as cursor:
                cursor.fast_executemany = True
                cursor.executemany(sql_query, product_list)
                conn.commit()
                print(f"Successfully bulk-inserted {len(product_list)} records.")
                
    except Error as e:
        print(f"Error during bulk insertion: {e}")

def insert_orders(order_list: List[Tuple[int, int, int, int, str, float]]) -> None:
    sql_query = "INSERT INTO Orders (OrderID, CustomerID, ProductID, Quantity, OrderDate, TotalAmount) VALUES (?, ?, ?, ?, ?, ?)"
    
    try:
        with get_db_connection() as conn:
            with conn.cursor() as cursor:
                cursor.fast_executemany = True
                cursor.executemany(sql_query, order_list)
                conn.commit()
                print(f"Successfully bulk-inserted {len(order_list)} records.")
                
    except Error as e:
        print(f"Error during bulk insertion: {e}")


if __name__ == "__main__":
   
    new_customers =[(1, 'John Smith', 'john.smith@email.com', 712345678),
        (2, 'Mary Johnson', 'mary.johnson@email.com', 723456789),
        (3, 'David Williams', 'david.williams@email.com', 734567890),
        (4, 'Sarah Brown', 'sarah.brown@email.com', 745678901),
        (5, 'Michael Jones', 'michael.jones@email.com', 756789012),
        (6, 'Linda Garcia', 'linda.garcia@email.com', 767890123),
        (7, 'James Miller', 'james.miller@email.com', 778901234),
        (8, 'Patricia Davis', 'patricia.davis@email.com', 789012345),
        (9, 'Robert Wilson', 'robert.wilson@email.com', 790123456),
        (10, 'Jennifer Moore', 'jennifer.moore@email.com', 701234567),
        (11, 'William Taylor', 'william.taylor@email.com', 711223344),
        (12, 'Elizabeth Anderson', 'elizabeth.anderson@email.com', 722334455),
        (13, 'Charles Thomas', 'charles.thomas@email.com', 733445566),
        (14, 'Susan Jackson', 'susan.jackson@email.com', 744556677),
        (15, 'Joseph White', 'joseph.white@email.com', 755667788),
        (16, 'Jessica Harris', 'jessica.harris@email.com', 766778899),
        (17, 'Thomas Martin', 'thomas.martin@email.com', 777889900),
        (18, 'Karen Thompson', 'karen.thompson@email.com', 788990011),
        (19, 'Daniel Martinez', 'daniel.martinez@email.com', 799001122),
        (20, 'Nancy Robinson', 'nancy.robinson@email.com', 700112233)]
    sql_query = "INSERT INTO Customers (CustomerID, CustomerName, Email, PhoneNumber) VALUES (?, ?, ?, ?)"
    insert_customers(sql_query, new_customers)

    new_products = [(1, 'Laptop', '15-inch Intel Core i7 laptop with 16GB RAM', 15999.99),
        (2, 'Wireless Mouse', 'Ergonomic Bluetooth wireless mouse', 349.99),
        (3, 'Mechanical Keyboard', 'RGB mechanical keyboard with blue switches', 899.99),
        (4, '24-inch Monitor', 'Full HD LED monitor', 2899.99),
        (5, 'USB Flash Drive', '64GB USB 3.0 flash drive', 199.99),
        (6, 'External Hard Drive', '1TB portable external hard drive', 1499.99),
        (7, 'Webcam', '1080p HD USB webcam', 699.99),
        (8, 'Office Chair', 'Ergonomic office chair with lumbar support', 2599.99),
        (9, 'Desk Lamp', 'LED desk lamp with adjustable brightness', 499.99),
        (10, 'Printer', 'Wireless all-in-one inkjet printer', 2299.99),
        (11, 'Smartphone', '128GB Android smartphone', 7999.99),
        (12, 'Tablet', '10-inch Android tablet', 4999.99),
        (13, 'Bluetooth Speaker', 'Portable waterproof Bluetooth speaker', 1199.99),
        (14, 'Wireless Earbuds', 'Noise-cancelling wireless earbuds', 1799.99),
        (15, 'Smart Watch', 'Fitness tracking smartwatch', 3499.99),
        (16, 'Power Bank', '20,000mAh fast charging power bank', 799.99),
        (17, 'Router', 'Dual-band Wi-Fi 6 router', 1899.99),
        (18, 'HDMI Cable', '2-meter high-speed HDMI cable', 149.99),
        (19, 'Graphics Tablet', 'Digital drawing tablet with stylus', 2699.99),
        (20, 'Microphone', 'USB condenser microphone for streaming', 1399.99),
        (21, 'Gaming Headset', 'Surround sound gaming headset with microphone', 1299.99),
        (22, 'SSD 512GB', '512GB SATA solid state drive', 999.99),
        (23, 'Laptop Stand', 'Adjustable aluminum laptop stand', 549.99),
        (24, 'Wireless Charger', '15W fast wireless charging pad', 399.99),
        (25, 'Projector', 'Full HD home theatre projector', 6499.99),
        (26, 'Scanner', 'High-speed document scanner', 3199.99),
        (27, 'Network Switch', '8-port Gigabit Ethernet switch', 899.99),
        (28, 'Surge Protector', '6-outlet surge protection power strip', 249.99),
        (29, 'Portable Fan', 'Rechargeable USB portable fan', 299.99),
        (30, 'Digital Camera', '24MP mirrorless digital camera', 12499.99)]
    
    insert_products(new_products)

    new_orders = [(1, 1, 1, 1, '2026-01-05 09:15:00', 15999.99),
            (2, 2, 5, 2, '2026-01-06 10:30:00', 399.98),
            (3, 3, 11, 1, '2026-01-07 14:20:00', 7999.99),
            (4, 4, 3, 1, '2026-01-08 11:45:00', 899.99),
            (5, 5, 8, 1, '2026-01-09 13:10:00', 2599.99),
            (6, 6, 14, 2, '2026-01-10 15:30:00', 3599.98),
            (7, 7, 2, 3, '2026-01-11 16:15:00', 1049.97),
            (8, 8, 20, 1, '2026-01-12 09:40:00', 1399.99),
            (9, 9, 18, 5, '2026-01-13 12:00:00', 749.95),
            (10, 10, 25, 1, '2026-01-14 17:25:00', 6499.99),
            (11, 11, 10, 1, '2026-01-15 08:50:00', 2299.99),
            (12, 12, 15, 2, '2026-01-16 11:35:00', 6999.98),
            (13, 13, 7, 1, '2026-01-17 13:55:00', 699.99),
            (14, 14, 22, 2, '2026-01-18 14:10:00', 1999.98),
            (15, 15, 17, 1, '2026-01-19 10:45:00', 1899.99),
            (16, 16, 9, 2, '2026-01-20 09:20:00', 999.98),
            (17, 17, 13, 1, '2026-01-21 15:40:00', 1199.99),
            (18, 18, 4, 1, '2026-01-22 16:30:00', 2899.99),
            (19, 19, 26, 1, '2026-01-23 12:50:00', 3199.99),
            (20, 20, 12, 1, '2026-01-24 11:05:00', 4999.99),
            (21, 1, 16, 2, '2026-01-25 10:15:00', 1599.98),
            (22, 2, 30, 1, '2026-01-26 14:45:00', 12499.99),
            (23, 3, 23, 2, '2026-01-27 15:30:00', 1099.98),
            (24, 4, 24, 3, '2026-01-28 09:50:00', 1199.97),
            (25, 5, 6, 1, '2026-01-29 16:10:00', 1499.99),
            (26, 6, 21, 1, '2026-01-30 13:25:00', 1299.99),
            (27, 7, 19, 1, '2026-02-01 11:35:00', 2699.99),
            (28, 8, 27, 2, '2026-02-02 12:20:00', 1799.98),
            (29, 9, 28, 4, '2026-02-03 14:00:00', 999.96),
            (30, 10, 29, 3, '2026-02-04 10:30:00', 899.97),
            (31, 11, 2, 1, '2026-02-05 09:15:00', 349.99),
            (32, 12, 5, 4, '2026-02-06 11:45:00', 799.96),
            (33, 13, 1, 1, '2026-02-07 15:20:00', 15999.99),
            (34, 14, 11, 2, '2026-02-08 16:40:00', 15999.98),
            (35, 15, 15, 1, '2026-02-09 10:00:00', 3499.99),
            (36, 16, 8, 2, '2026-02-10 13:50:00', 5199.98),
            (37, 17, 9, 1, '2026-02-11 12:10:00', 499.99),
            (38, 18, 14, 3, '2026-02-12 14:25:00', 5399.97),
            (39, 19, 22, 2, '2026-02-13 11:55:00', 1999.98),
            (40, 20, 25, 1, '2026-02-14 16:05:00', 6499.99),
            (41, 1, 3, 2, '2026-02-15 09:40:00', 1799.98),
            (42, 2, 18, 6, '2026-02-16 13:15:00', 899.94),
            (43, 3, 13, 2, '2026-02-17 15:45:00', 2399.98),
            (44, 4, 17, 1, '2026-02-18 10:50:00', 1899.99),
            (45, 5, 20, 2, '2026-02-19 12:30:00', 2799.98),
            (46, 6, 24, 1, '2026-02-20 14:35:00', 399.99),
            (47, 7, 26, 1, '2026-02-21 11:20:00', 3199.99),
            (48, 8, 10, 2, '2026-02-22 16:15:00', 4599.98),
            (49, 9, 30, 1, '2026-02-23 09:10:00', 12499.99),
            (50, 10, 6, 2, '2026-02-24 13:40:00', 2999.98)]
    
    insert_orders(new_orders)


Successfully bulk-inserted 20 records.
Successfully bulk-inserted 30 records.
Successfully bulk-inserted 50 records.


In [3]:
# 1. UPDATE CUSTOMER EMAIL

SERVER = "localhost\\SQLEXPRESS"
DRIVER = "SQL Server"
DATABASE = "SalesDB"

def get_db_connection() -> pyodbc.Connection:
    conn_str = (
        f"DRIVER={{{DRIVER}}};"
        f"SERVER={SERVER};"
        f"DATABASE={DATABASE};"
        f"Trusted_Connection=yes;"
    )
    return pyodbc.connect(conn_str)

def update_customer_email(customer_id: int, new_email: str) -> None:
    """Updates the email address of a specific customer using their ID."""
    sql = "UPDATE Customers SET Email = ? WHERE CustomerID = ?"
    
    try:
        with get_db_connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute(sql, (new_email, customer_id))
                conn.commit()
                print(f"Successfully updated email for CustomerID {customer_id}.")
    except Error as e:
        print(f"Error updating customer email: {e}")

if __name__ == "__main__":

    update_customer_email(customer_id=1, new_email="new_contact@example.com")
 


Successfully updated email for CustomerID 1.


In [ ]:
# 2. UPDATE PRODUCT PRICE

SERVER = "localhost\\SQLEXPRESS"
DRIVER = "SQL Server"
DATABASE = "SalesDB"

def get_db_connection() -> pyodbc.Connection:
    conn_str = (
        f"DRIVER={{{DRIVER}}};"
        f"SERVER={SERVER};"
        f"DATABASE={DATABASE};"
        f"Trusted_Connection=yes;"
    )
    return pyodbc.connect(conn_str)

def update_product_price(product_id: int, new_price: float) -> None:
    """Updates the price of a specific product using its ID."""
    sql = "UPDATE Products SET Price = ? WHERE ProductID = ?"
    
    try:
        with get_db_connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute(sql, (new_price, product_id))
                conn.commit()
                print(f"Successfully updated price for ProductID {product_id} to R{new_price:.2f}.")
    except Error as e:
        print(f"Error updating product price: {e}")




if __name__ == "__main__":
 
    update_product_price(product_id=5, new_price=1000.00)
   

Successfully updated price for ProductID 5 to R1000.00.


In [2]:
# 3. DELETE CUSTOMER

SERVER = "localhost\\SQLEXPRESS"
DRIVER = "SQL Server"
DATABASE = "SalesDB"

def get_db_connection() -> pyodbc.Connection:
    conn_str = (
        f"DRIVER={{{DRIVER}}};"
        f"SERVER={SERVER};"
        f"DATABASE={DATABASE};"
        f"Trusted_Connection=yes;"
    )
    return pyodbc.connect(conn_str)

def delete_customer(customer_id: int) -> None:
    """Deletes a customer profile entirely using their ID."""
    sql = "DELETE FROM Customers WHERE CustomerID = ?"
    
    try:
        with get_db_connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute(sql, (customer_id,))
                conn.commit()
                print(f"Successfully deleted CustomerID {customer_id}.")
    except Error as e:
        print(f"Error deleting customer: {e}")

if __name__ == "__main__":
    delete_customer(customer_id=20)
    


Error deleting customer: ('23000', '[23000] [Microsoft][ODBC SQL Server Driver][SQL Server]The DELETE statement conflicted with the REFERENCE constraint "FK__Orders__Customer__60A75C0F". The conflict occurred in database "SalesDB", table "dbo.Orders", column \'CustomerID\'. (547) (SQLExecDirectW); [23000] [Microsoft][ODBC SQL Server Driver][SQL Server]The statement has been terminated. (3621)')


In [138]:
#4. READ PRODUCTS

import sql
import pandas as pd
from sqlalchemy import create_engine



# def get_db_connection(conn_str) -> pyodbc.Connection:
#     # conn = pyodbc.connect(conn_str)
#     # return conn

SERVER = "localhost\\SQLEXPRESS"
DRIVER = "SQL Server"
DATABASE = "SalesDB"

def get_db_connection() -> pyodbc.Connection:
    conn_str = (
        f"DRIVER={{{DRIVER}}};"
        f"SERVER={SERVER};"
        f"DATABASE={DATABASE};"
        f"Trusted_Connection=yes;"
    )
    return pyodbc.connect(conn_str)
def read_query_df(sql: str) -> pd.DataFrame:
    """
    Fetches all records from the Products table and returns them
    directly as a structured pandas DataFrame.
    """
    # sql = "SELECT * FROM Products"
    products_df = pd.DataFrame()
     
    try:
        with get_db_connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute(sql)
                rows = cursor.fetchall()
                columns = [column[0] for column in cursor.description]
                products_df = pd.DataFrame.from_records(rows, columns=columns)
                
    except Error as e:
        print(f"Error fetching products: {e}")

    return products_df

def read_data(sql_query, conn):
    try:
        df = pd.read_sql(sql_query, conn)
            
    except Error as e:
            print(f"Error fetching products: {e}")
    
    return df

import warnings


warnings.filterwarnings(

    "ignore",

    message="pandas only supports SQLAlchemy connectable"

)
def sqlalchamy_conn(server, database, driver="ODBC Driver 17 for SQL Server"):
    """
    Establishes a connection to the SQL Server database using SQLAlchemy.
    """
    connection_url = f"mssql+pyodbc://@{server}/{database}?driver={driver}&trusted_connection=yes"
    engine = create_engine(connection_url)
    conn = engine.connect()
    return conn

# if __name__ == "__main__" :
    
SERVER = "localhost\\SQLEXPRESS"
DRIVER = "SQL Server"
DATABASE = "SalesDB"
query = "SELECT * FROM Products"
# conn_str = (
#         f"DRIVER={{{DRIVER}}};"
#         f"SERVER={SERVER};"
#         f"DATABASE={DATABASE};"
#         f"Trusted_Connection=yes;"
#     )
    # conn = get_db_connection(conn_str)
# # df = read_data(query, conn)
# print(df)
conn = sqlalchamy_conn(SERVER, DATABASE, driver="ODBC Driver 17 for SQL Server")
df = read_data(query, conn)
df

,ProductID,ProductName,Description,Price
0,1,Laptop,15-inch Intel Core i7 laptop with 16GB RAM,15999.99
1,2,Wireless Mouse,Ergonomic Bluetooth wireless mouse,349.99
2,3,Mechanical Keyboard,RGB mechanical keyboard with blue switches,899.99
3,4,24-inch Monitor,Full HD LED monitor,2899.99
4,5,USB Flash Drive,64GB USB 3.0 flash drive,1000.00
5,6,External Hard Drive,1TB portable external hard drive,1499.99
6,7,Webcam,1080p HD USB webcam,699.99
7,8,Office Chair,Ergonomic office chair with lumbar support,2599.99
8,9,Desk Lamp,LED desk lamp with adjustable brightness,499.99
9,10,Printer,Wireless all-in-one inkjet printer,2299.99


In [122]:
df

,ProductID,ProductName,Description,Price
0,1,Laptop,15-inch Intel Core i7 laptop with 16GB RAM,15999.99
1,2,Wireless Mouse,Ergonomic Bluetooth wireless mouse,349.99
2,3,Mechanical Keyboard,RGB mechanical keyboard with blue switches,899.99
3,4,24-inch Monitor,Full HD LED monitor,2899.99
4,5,USB Flash Drive,64GB USB 3.0 flash drive,1000.00
5,6,External Hard Drive,1TB portable external hard drive,1499.99
6,7,Webcam,1080p HD USB webcam,699.99
7,8,Office Chair,Ergonomic office chair with lumbar support,2599.99
8,9,Desk Lamp,LED desk lamp with adjustable brightness,499.99
9,10,Printer,Wireless all-in-one inkjet printer,2299.99


##Testing of Queries

In [86]:
sql = "SELECT * FROM Products"
df = read_query_df(sql)
df


,ProductID,ProductName,Description,Price
0,1,Laptop,15-inch Intel Core i7 laptop with 16GB RAM,15999.99
1,2,Wireless Mouse,Ergonomic Bluetooth wireless mouse,349.99
2,3,Mechanical Keyboard,RGB mechanical keyboard with blue switches,899.99
3,4,24-inch Monitor,Full HD LED monitor,2899.99
4,5,USB Flash Drive,64GB USB 3.0 flash drive,1000.00
5,6,External Hard Drive,1TB portable external hard drive,1499.99
6,7,Webcam,1080p HD USB webcam,699.99
7,8,Office Chair,Ergonomic office chair with lumbar support,2599.99
8,9,Desk Lamp,LED desk lamp with adjustable brightness,499.99
9,10,Printer,Wireless all-in-one inkjet printer,2299.99


In [92]:
sql = "SELECT * FROM Customers"
df = read_query_df(sql)
df

,CustomerID,CustomerName,Email,PhoneNumber
0,1,John Smith,new_contact@example.com,712345678
1,2,Mary Johnson,mary.johnson@email.com,723456789
2,3,David Williams,david.williams@email.com,734567890
3,4,Sarah Brown,sarah.brown@email.com,745678901
4,5,Michael Jones,michael.jones@email.com,756789012
5,6,Linda Garcia,linda.garcia@email.com,767890123
6,7,James Miller,james.miller@email.com,778901234
7,8,Patricia Davis,patricia.davis@email.com,789012345
8,9,Robert Wilson,robert.wilson@email.com,790123456
9,10,Jennifer Moore,jennifer.moore@email.com,701234567


In [93]:
sql = "SELECT * FROM Orders"
df = read_query_df(sql)
df

,OrderID,CustomerID,ProductID,Quantity,OrderDate,TotalAmount
0,1,1,1,1,2026-01-05 09:15:00,15999.99
1,2,2,5,2,2026-01-06 10:30:00,399.98
2,3,3,11,1,2026-01-07 14:20:00,7999.99
3,4,4,3,1,2026-01-08 11:45:00,899.99
4,5,5,8,1,2026-01-09 13:10:00,2599.99
5,6,6,14,2,2026-01-10 15:30:00,3599.98
6,7,7,2,3,2026-01-11 16:15:00,1049.97
7,8,8,20,1,2026-01-12 09:40:00,1399.99
8,9,9,18,5,2026-01-13 12:00:00,749.95
9,10,10,25,1,2026-01-14 17:25:00,6499.99


In [87]:
sql = "SELECT * FROM Products ORDER BY Price DESC"
df = read_query_df(sql)
df

,ProductID,ProductName,Description,Price
0,1,Laptop,15-inch Intel Core i7 laptop with 16GB RAM,15999.99
1,30,Digital Camera,24MP mirrorless digital camera,12499.99
2,11,Smartphone,128GB Android smartphone,7999.99
3,25,Projector,Full HD home theatre projector,6499.99
4,12,Tablet,10-inch Android tablet,4999.99
5,15,Smart Watch,Fitness tracking smartwatch,3499.99
6,26,Scanner,High-speed document scanner,3199.99
7,4,24-inch Monitor,Full HD LED monitor,2899.99
8,19,Graphics Tablet,Digital drawing tablet with stylus,2699.99
9,8,Office Chair,Ergonomic office chair with lumbar support,2599.99


In [89]:
sql = "SELECT * FROM Customers WHERE CustomerName LIKE '[a-f]%'"
df = read_query_df(sql)
df

,CustomerID,CustomerName,Email,PhoneNumber
0,3,David Williams,david.williams@email.com,734567890
1,12,Elizabeth Anderson,elizabeth.anderson@email.com,722334455
2,13,Charles Thomas,charles.thomas@email.com,733445566
3,19,Daniel Martinez,daniel.martinez@email.com,799001122


In [90]:
sql = "select CustomerID AS ID, CustomerName AS Customer FROM Customers"
df = read_query_df(sql)
df

,ID,Customer
0,1,John Smith
1,2,Mary Johnson
2,3,David Williams
3,4,Sarah Brown
4,5,Michael Jones
5,6,Linda Garcia
6,7,James Miller
7,8,Patricia Davis
8,9,Robert Wilson
9,10,Jennifer Moore


In [91]:
sql = "select Orders.OrderID, Customers.CustomerName, Orders.OrderDate FROM Orders INNER JOIN Customers ON Orders.CustomerID=Customers.CustomerID"
df = read_query_df(sql)
df

,OrderID,CustomerName,OrderDate
0,1,John Smith,2026-01-05 09:15:00
1,2,Mary Johnson,2026-01-06 10:30:00
2,3,David Williams,2026-01-07 14:20:00
3,4,Sarah Brown,2026-01-08 11:45:00
4,5,Michael Jones,2026-01-09 13:10:00
5,6,Linda Garcia,2026-01-10 15:30:00
6,7,James Miller,2026-01-11 16:15:00
7,8,Patricia Davis,2026-01-12 09:40:00
8,9,Robert Wilson,2026-01-13 12:00:00
9,10,Jennifer Moore,2026-01-14 17:25:00


In [94]:
sql = "select Orders.OrderID, Customers.CustomerName, Orders.OrderDate FROM Orders LEFT JOIN Customers ON Orders.CustomerID=Customers.CustomerID"
df = read_query_df(sql)
df

,OrderID,CustomerName,OrderDate
0,1,John Smith,2026-01-05 09:15:00
1,2,Mary Johnson,2026-01-06 10:30:00
2,3,David Williams,2026-01-07 14:20:00
3,4,Sarah Brown,2026-01-08 11:45:00
4,5,Michael Jones,2026-01-09 13:10:00
5,6,Linda Garcia,2026-01-10 15:30:00
6,7,James Miller,2026-01-11 16:15:00
7,8,Patricia Davis,2026-01-12 09:40:00
8,9,Robert Wilson,2026-01-13 12:00:00
9,10,Jennifer Moore,2026-01-14 17:25:00


In [95]:
sql = "select Orders.OrderID, Customers.CustomerName, Orders.OrderDate FROM Orders RIGHT JOIN Customers ON Orders.CustomerID=Customers.CustomerID"
df = read_query_df(sql)
df

,OrderID,CustomerName,OrderDate
0,1,John Smith,2026-01-05 09:15:00
1,21,John Smith,2026-01-25 10:15:00
2,41,John Smith,2026-02-15 09:40:00
3,2,Mary Johnson,2026-01-06 10:30:00
4,22,Mary Johnson,2026-01-26 14:45:00
5,42,Mary Johnson,2026-02-16 13:15:00
6,3,David Williams,2026-01-07 14:20:00
7,23,David Williams,2026-01-27 15:30:00
8,43,David Williams,2026-02-17 15:45:00
9,4,Sarah Brown,2026-01-08 11:45:00


In [96]:
sql = "select Orders.OrderID, Customers.CustomerName, Orders.OrderDate FROM Orders FULL JOIN Customers ON Orders.CustomerID=Customers.CustomerID WHERE Customers.CustomerID IS NOT NULL"
df = read_query_df(sql)
df

,OrderID,CustomerName,OrderDate
0,1,John Smith,2026-01-05 09:15:00
1,21,John Smith,2026-01-25 10:15:00
2,41,John Smith,2026-02-15 09:40:00
3,2,Mary Johnson,2026-01-06 10:30:00
4,22,Mary Johnson,2026-01-26 14:45:00
5,42,Mary Johnson,2026-02-16 13:15:00
6,3,David Williams,2026-01-07 14:20:00
7,23,David Williams,2026-01-27 15:30:00
8,43,David Williams,2026-02-17 15:45:00
9,4,Sarah Brown,2026-01-08 11:45:00


Business Questions
Which customers placed orders?
Which customers have never ordered?
What products were purchased in each order?

In [97]:
sql = "select * FROM Customers WHERE CustomerID IN (select CustomerID FROM Orders)"
df = read_query_df(sql)
df


,CustomerID,CustomerName,Email,PhoneNumber
0,1,John Smith,new_contact@example.com,712345678
1,2,Mary Johnson,mary.johnson@email.com,723456789
2,3,David Williams,david.williams@email.com,734567890
3,4,Sarah Brown,sarah.brown@email.com,745678901
4,5,Michael Jones,michael.jones@email.com,756789012
5,6,Linda Garcia,linda.garcia@email.com,767890123
6,7,James Miller,james.miller@email.com,778901234
7,8,Patricia Davis,patricia.davis@email.com,789012345
8,9,Robert Wilson,robert.wilson@email.com,790123456
9,10,Jennifer Moore,jennifer.moore@email.com,701234567


In [98]:
sql = "select * FROM Customers WHERE CustomerID NOT IN (select CustomerID FROM Orders)"
df = read_query_df(sql)
df

,CustomerID,CustomerName,Email,PhoneNumber


In [99]:
sql = "select * FROM Products WHERE ProductID IN (select ProductID FROM Orders)"
df = read_query_df(sql)
df

,ProductID,ProductName,Description,Price
0,1,Laptop,15-inch Intel Core i7 laptop with 16GB RAM,15999.99
1,2,Wireless Mouse,Ergonomic Bluetooth wireless mouse,349.99
2,3,Mechanical Keyboard,RGB mechanical keyboard with blue switches,899.99
3,4,24-inch Monitor,Full HD LED monitor,2899.99
4,5,USB Flash Drive,64GB USB 3.0 flash drive,1000.00
5,6,External Hard Drive,1TB portable external hard drive,1499.99
6,7,Webcam,1080p HD USB webcam,699.99
7,8,Office Chair,Ergonomic office chair with lumbar support,2599.99
8,9,Desk Lamp,LED desk lamp with adjustable brightness,499.99
9,10,Printer,Wireless all-in-one inkjet printer,2299.99


6/8/2026
Reports
Total Revenue
Average Order Value
Orders per Customer
Top Selling Product


In [175]:
#Total Revenue
# sql = "SELECT * FROM Orders"
sql = "SELECT SUM(Quantity * TotalAmount) AS TotalRevenue FROM Orders"
df = read_query_df(sql)
df

,TotalRevenue
0,270047.81


In [146]:
#Average Order Value
sql = "SELECT AVG(Quantity * TotalAmount) AS AverageOrderValue FROM Orders"
df = read_query_df(sql)
df

,AverageOrderValue
0,5400.956200


In [176]:
print("Orders per Customer:")
#Orders per Customer
sql = "SELECT CustomerID, COUNT(*) AS OrdersPerCustomer FROM Orders GROUP BY CustomerID"
df = read_query_df(sql)
df

Orders per Customer:


,CustomerID,OrdersPerCustomer
0,1,3
1,2,3
2,3,3
3,4,3
4,5,3
5,6,3
6,7,3
7,8,3
8,9,3
9,10,3


In [160]:
#Top Selling Product
sql = "SELECT ProductID, SUM(Quantity) AS TotalQuantity FROM Orders GROUP BY ProductID HAVING SUM(Quantity) = (SELECT MAX(TotalQuantity) FROM (SELECT SUM(Quantity) AS TotalQuantity FROM Orders GROUP BY ProductID) AS Subquery)"
df = read_query_df(sql)
df

,ProductID,TotalQuantity
0,18,11


In [ ]:
#Top Selling Product( Method 2- select Top)
sql = "SELECT TOP 1 ProductID, SUM(Quantity) AS TotalQuantity FROM Orders GROUP BY ProductID ORDER BY TotalQuantity DESC"
df = read_query_df(sql)
df

,ProductID,TotalQuantity
0,18,11


Implement:

    Single-row Subqueries
    Multi-row Subqueries
    Correlated Subqueries

Business Questions

    Products above average price
    Highest-spending customer
    Orders greater than average order value

In [167]:
#Single-row Subqueries
#Products above average price

sql = "SELECT * FROM Products WHERE Price > (SELECT AVG(Price) FROM Products)"
df = read_query_df(sql)
df

,ProductID,ProductName,Description,Price
0,1,Laptop,15-inch Intel Core i7 laptop with 16GB RAM,15999.99
1,4,24-inch Monitor,Full HD LED monitor,2899.99
2,11,Smartphone,128GB Android smartphone,7999.99
3,12,Tablet,10-inch Android tablet,4999.99
4,15,Smart Watch,Fitness tracking smartwatch,3499.99
5,25,Projector,Full HD home theatre projector,6499.99
6,26,Scanner,High-speed document scanner,3199.99
7,30,Digital Camera,24MP mirrorless digital camera,12499.99


In [184]:
#Highest-spending customer
# sql = "SELECT * FROM Orders"
sql = "SELECT * FROM Customers WHERE CustomerID IN (SELECT CustomerID FROM Orders WHERE Quantity > 1)"
df = read_query_df(sql)
df

,CustomerID,CustomerName,Email,PhoneNumber
0,1,John Smith,new_contact@example.com,712345678
1,2,Mary Johnson,mary.johnson@email.com,723456789
2,3,David Williams,david.williams@email.com,734567890
3,4,Sarah Brown,sarah.brown@email.com,745678901
4,5,Michael Jones,michael.jones@email.com,756789012
5,6,Linda Garcia,linda.garcia@email.com,767890123
6,7,James Miller,james.miller@email.com,778901234
7,8,Patricia Davis,patricia.davis@email.com,789012345
8,9,Robert Wilson,robert.wilson@email.com,790123456
9,10,Jennifer Moore,jennifer.moore@email.com,701234567


sql = "SELECT * FROM Customers WHERE CustomerID IN (SELECT TOP 1  CustomerID FROM Orders GROUP BY CustomerID ORDER BY SUM(TotalAmount) DESC)"

In [173]:
#Orders greater than average order value
sql = "SELECT * FROM Orders WHERE TotalAmount > (SELECT AVG(TotalAmount) FROM Orders)"
df = read_query_df(sql)
df

,OrderID,CustomerID,ProductID,Quantity,OrderDate,TotalAmount
0,1,1,1,1,2026-01-05 09:15:00,15999.99
1,3,3,11,1,2026-01-07 14:20:00,7999.99
2,10,10,25,1,2026-01-14 17:25:00,6499.99
3,12,12,15,2,2026-01-16 11:35:00,6999.98
4,20,20,12,1,2026-01-24 11:05:00,4999.99
5,22,2,30,1,2026-01-26 14:45:00,12499.99
6,33,13,1,1,2026-02-07 15:20:00,15999.99
7,34,14,11,2,2026-02-08 16:40:00,15999.98
8,36,16,8,2,2026-02-10 13:50:00,5199.98
9,38,18,14,3,2026-02-12 14:25:00,5399.97


Do visualizations
write reports on what youve done
research on github